# Step 1: 3-way vs 2-way vs FedAvg vs Independent

**Claim**: 3-way decomposition (E + θ_task + θ_BS) outperforms 2-way (FedPer), FedAvg, and independent training.

## Methods compared:
1. **3-way (Ours)**: E + θ_task shared via FL, θ_BS local
2. **FedPer (2-way)**: E shared, θ_task local
3. **FedAvg**: all params shared
4. **Independent**: per-BS training, no sharing
5. **Centralized**: all data pooled (upper bound)

In [ ]:
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.config import DatasetConfig, TrainConfig
from src.data.dataset import ChannelEstimationDataset, PerBSDataLoader
from src.models.estimator import create_model
from src.models.baselines import PlainEstimator, FedPerEstimator
from src.training.federated import federated_train
from src.training.trainer import train_local, evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
DATA_DIR = '../../data/channels'
ALL_BS = list(range(8))
SNR_RANGE = (0.0, 30.0)
BATCH_SIZE = 32
FL_ROUNDS = 50
LOCAL_EPOCHS = 5

In [ ]:
# Per-BS data loaders
train_loaders = {}
val_loaders = {}
for bs_id in ALL_BS:
    ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[bs_id], snr_range_db=SNR_RANGE)
    n = len(ds)
    n_val = max(int(n * 0.2), 1)
    n_train = n - n_val
    train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, n_val])
    train_loaders[bs_id] = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loaders[bs_id] = DataLoader(val_ds, batch_size=BATCH_SIZE)
    print(f'BS{bs_id}: {n_train} train, {n_val} val')

## Method 1: 3-way (Ours)

In [ ]:
def model_3way():
    return create_model(site_integration='film', site_embed_dim=64)

result_3way = federated_train(
    model_fn=model_3way,
    train_loaders=train_loaders,
    val_loaders=val_loaders,
    fl_rounds=FL_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    device=device,
)

## Method 2: FedPer (2-way)

In [ ]:
def model_fedper():
    return FedPerEstimator()

result_fedper = federated_train(
    model_fn=model_fedper,
    train_loaders=train_loaders,
    val_loaders=val_loaders,
    fl_rounds=FL_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    device=device,
)

## Method 3: FedAvg

In [ ]:
def model_plain():
    return PlainEstimator()

result_fedavg = federated_train(
    model_fn=model_plain,
    train_loaders=train_loaders,
    val_loaders=val_loaders,
    fl_rounds=FL_ROUNDS,
    local_epochs=LOCAL_EPOCHS,
    device=device,
)

## Method 4: Independent

In [ ]:
result_indep = {}
for bs_id in ALL_BS:
    print(f'\nTraining independent model for BS{bs_id}...')
    model = PlainEstimator().to(device)
    res = train_local(
        model, train_loaders[bs_id], val_loaders[bs_id],
        epochs=FL_ROUNDS * LOCAL_EPOCHS,  # Same total epochs
        lr=1e-3, device=device,
    )
    result_indep[bs_id] = {'model': model, 'result': res}

## Compare Results

In [ ]:
# Final per-BS NMSE comparison
methods = {'3-way (Ours)': result_3way, 'FedPer': result_fedper, 'FedAvg': result_fedavg}
final_nmse = {name: {} for name in methods}
final_nmse['Independent'] = {}

for name, result in methods.items():
    for bs_id in ALL_BS:
        _, db = evaluate(result['models'][bs_id], val_loaders[bs_id], device)
        final_nmse[name][bs_id] = db

for bs_id in ALL_BS:
    _, db = evaluate(result_indep[bs_id]['model'], val_loaders[bs_id], device)
    final_nmse['Independent'][bs_id] = db

# Bar plot
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(ALL_BS))
width = 0.2
for i, (name, vals) in enumerate(final_nmse.items()):
    ax.bar(x + i*width, [vals[bs] for bs in ALL_BS], width, label=name)

ax.set_xlabel('BS ID')
ax.set_ylabel('NMSE (dB)')
ax.set_title('Per-BS Channel Estimation Performance')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels([f'BS{i}' for i in ALL_BS])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('step1_comparison.png', dpi=150)
plt.show()

In [ ]:
# FL convergence curves
fig, ax = plt.subplots(figsize=(8, 5))
for name, result in methods.items():
    avg_per_round = [
        np.mean([result['history']['val_nmse_db'][bs][r]
                 for bs in ALL_BS if result['history']['val_nmse_db'][bs][r] is not None])
        for r in range(len(result['history']['round']))
    ]
    ax.plot(avg_per_round, label=name, linewidth=2)

ax.set_xlabel('FL Round')
ax.set_ylabel('Avg NMSE (dB)')
ax.set_title('FL Convergence Comparison')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step1_convergence.png', dpi=150)
plt.show()